# RAG System on Google Colab with GPU (T4)
## Complete Setup and Execution

This notebook sets up a complete Retrieval-Augmented Generation system using:
- **Hugging Face models** with GPU acceleration
- **T4 GPU** for fast inference
- **Flask web UI** exposed via ngrok
- **ChromaDB** for vector storage

### Prerequisites:
1. A PDF file to analyze (upload or use example)
2. ngrok auth token (get free from https://ngrok.com)


## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q chromadb sentence-transformers transformers
!pip install -q flask pyngrok
!pip install -q pypdf

print("✓ All dependencies installed successfully!")

## Step 2: Check GPU and Setup

In [ ]:
import torch
import os
from google.colab import drive

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Create working directories
os.makedirs('uploads', exist_ok=True)
os.makedirs('chroma_db', exist_ok=True)

print("\n✓ GPU setup complete!")

## Step 3: Setup ngrok Tunnel

In [ ]:
from pyngrok import ngrok

# Get your auth token from https://ngrok.com
NGROK_AUTH_TOKEN = ""  # REPLACE THIS WITH YOUR NGROK TOKEN

if not NGROK_AUTH_TOKEN:
    print("⚠️  IMPORTANT: Please update NGROK_AUTH_TOKEN above!")
    print("\nSteps to get a free ngrok token:")
    print("1. Go to https://ngrok.com/signup")
    print("2. Sign up for free")
    print("3. Go to https://dashboard.ngrok.com/auth/your-authtoken")
    print("4. Copy your auth token")
    print("5. Replace the empty string above with your token")
    print("6. Re-run this cell")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✓ ngrok configured!")

## Step 4: Create RAG System Code

In [ ]:
# Create book_qa_colab.py
book_qa_code = '''
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
import torch

# Configuration
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"  # Lightweight model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # Fast embeddings
CHUNK_SIZE = 500
CHUNK_OVERLAP = 150
VECTOR_DB_PATH = "./chroma_db"
RETRIEVAL_K = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

def load_and_chunk_book(pdf_path: str):
    """Load PDF and split into chunks"""
    print(f"Loading book from {pdf_path}...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\\n\\n", "\\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks

def create_vector_store(chunks):
    """Create vector store with embeddings"""
    print(f"Creating embeddings...")
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": DEVICE}
    )
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=VECTOR_DB_PATH,
        collection_name="book_qa"
    )
    print("Vector store created successfully")
    return vector_store

def load_vector_store():
    """Load existing vector store"""
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": DEVICE}
    )
    vector_store = Chroma(
        persist_directory=VECTOR_DB_PATH,
        embedding_function=embeddings,
        collection_name="book_qa"
    )
    return vector_store

def create_rag_chain(vector_store):
    """Create RAG chain"""
    print(f"Initializing LLM...")
    llm = HuggingFacePipeline(
        model_id=LLM_MODEL,
        task="text-generation",
        pipeline_kwargs={
            "torch_dtype": torch.float16,
            "device": DEVICE,
            "max_length": 512,
            "temperature": 0.7,
        }
    )
    
    prompt_template = """Use the following context to answer the question clearly and comprehensively.

Format your answer with:
- Paragraphs for explanations
- Bullet points for lists
- Numbered lists for steps
- **bold** for important terms
- ## Headers for sections

Context:
{context}

Question: {question}

Answer:"""
    
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["context", "question"]
    )
    
    retriever = vector_store.as_retriever(search_kwargs={"k": RETRIEVAL_K})
    chain = (
        RunnableParallel(
            context=retriever | (lambda docs: "\\n\\n".join([doc.page_content for doc in docs])),
            question=RunnablePassthrough()
        )
        | prompt
        | llm
        | StrOutputParser()
    )
    
    class RAGChain:
        def __init__(self, chain, retriever):
            self.chain = chain
            self.retriever = retriever
        
        def invoke(self, input_dict):
            question = input_dict.get("query", "")
            answer = self.chain.invoke(question)
            source_documents = self.retriever.invoke(question)
            return {"result": answer, "source_documents": source_documents}
    
    return RAGChain(chain, retriever)

def setup_rag(pdf_path: str):
    chunks = load_and_chunk_book(pdf_path)
    vector_store = create_vector_store(chunks)
    qa_chain = create_rag_chain(vector_store)
    return qa_chain

def load_rag():
    vector_store = load_vector_store()
    qa_chain = create_rag_chain(vector_store)
    return qa_chain
'''

with open('book_qa_colab.py', 'w') as f:
    f.write(book_qa_code)

print("✓ RAG system code created!")

## Step 5: Create Flask Web App

In [ ]:
# Create web_app_colab.py with complete PDF serving support
web_app_code = '''
from flask import Flask, render_template, request, jsonify, send_file
import os
from werkzeug.utils import secure_filename
from book_qa_colab import create_rag_chain, load_vector_store, create_vector_store, load_and_chunk_book

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 100 * 1024 * 1024
app.config["UPLOAD_FOLDER"] = "uploads"

os.makedirs(app.config["UPLOAD_FOLDER"], exist_ok=True)

qa_chain = None
vector_store_loaded = False
ALLOWED_EXTENSIONS = {"pdf"}

def allowed_file(filename):
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS

@app.route("/")
def index():
    html = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Book QA - GPU Powered</title>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/pdf.js/3.11.174/pdf.min.js"></script>
    <script>pdfjsLib.GlobalWorkerOptions.workerSrc = "https://cdnjs.cloudflare.com/ajax/libs/pdf.js/3.11.174/pdf.worker.min.js";</script>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto; background: #0f172a; color: #e2e8f0; }
        .container { max-width: 1200px; margin: 0 auto; padding: 20px; }
        .header { background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); padding: 30px 0; border-bottom: 1px solid #334155; }
        .header h1 { font-size: 32px; margin-bottom: 10px; }
        .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 20px; }
        .card { background: #1e293b; border: 1px solid #334155; border-radius: 12px; padding: 20px; }
        .upload-area { border: 2px dashed #475569; border-radius: 8px; padding: 40px; text-align: center; cursor: pointer; }
        .upload-area:hover { border-color: #3b82f6; background: rgba(59, 130, 246, 0.05); }
        .upload-area.dragover { border-color: #3b82f6; background: rgba(59, 130, 246, 0.1); }
        input[type="file"] { display: none; }
        .file-info { margin: 15px 0; padding: 10px; background: #0f172a; border-radius: 6px; }
        .btn { padding: 10px 20px; background: #3b82f6; color: white; border: none; border-radius: 6px; cursor: pointer; }
        .btn:hover { background: #2563eb; }
        .btn:disabled { background: #64748b; cursor: not-allowed; }
        .chat-messages { height: 300px; overflow-y: auto; margin-bottom: 15px; padding: 15px; background: #0f172a; border-radius: 8px; }
        .message { margin: 10px 0; padding: 10px; border-radius: 6px; }
        .message.user { background: #3b82f6; margin-left: 20px; }
        .message.bot { background: #334155; margin-right: 20px; }
        .input-area { display: flex; gap: 10px; }
        .input-area input { flex: 1; padding: 10px; background: #0f172a; border: 1px solid #475569; border-radius: 6px; color: #e2e8f0; }
        .status { padding: 10px; border-radius: 6px; margin: 10px 0; }
        .status.success { background: #064e3b; color: #d1fae5; }
        .status.error { background: #7c2d12; color: #fed7aa; }
        .status.processing { background: #78350f; color: #fef3c7; }
        .source-card { background: #0f172a; border: 1px solid #334155; border-radius: 8px; padding: 12px; margin: 10px 0; }
        .source-page { background: #3b82f6; color: white; padding: 4px 8px; border-radius: 4px; font-size: 12px; font-weight: bold; display: inline-block; margin-bottom: 8px; cursor: pointer; }
        .source-page:hover { background: #2563eb; }
        .pdf-modal { display: none; position: fixed; top: 50%; left: 50%; transform: translate(-50%, -50%); background: #1e293b; border: 1px solid #334155; border-radius: 12px; z-index: 1000; width: 90%; max-width: 800px; max-height: 90vh; overflow: auto; }
        .pdf-modal.show { display: block; }
        .pdf-modal-header { padding: 15px; border-bottom: 1px solid #334155; display: flex; justify-content: space-between; align-items: center; }
        .pdf-modal-close { background: none; border: none; color: #e2e8f0; font-size: 24px; cursor: pointer; }
        .pdf-modal-body { padding: 15px; max-height: 70vh; overflow-y: auto; }
        .pdf-overlay { display: none; position: fixed; top: 0; left: 0; right: 0; bottom: 0; background: rgba(0, 0, 0, 0.7); z-index: 999; }
        .pdf-overlay.show { display: block; }
    </style>
</head>
<body>
    <header class="header">
        <div class="container">
            <h1>📚 Book QA - GPU Powered</h1>
            <p>Running on Google Colab with T4 GPU</p>
        </div>
    </header>
    <main class="container">
        <div class="grid">
            <section class="card">
                <h2>📄 Upload PDF</h2>
                <div class="upload-area" id="uploadArea" ondrop="handleDrop(event)" ondragover="event.preventDefault(); uploadArea.classList.add('dragover')" ondragleave="uploadArea.classList.remove('dragover')">
                    <h3>Drag & Drop PDF</h3>
                    <p>or click to select</p>
                    <input type="file" id="fileInput" accept=".pdf" onchange="handleFileSelect(event)">
                </div>
                <div class="file-info" id="fileInfo" style="display:none">
                    <p><strong>File:</strong> <span id="fileName"></span></p>
                    <p><strong>Status:</strong> <span id="fileStatus"></span></p>
                </div>
                <button class="btn" id="processBtn" onclick="processFile()" style="width:100%; display:none; margin-top:10px">Process Book</button>
                <div id="statusBox"></div>
            </section>
            <section class="card">
                <h2>❓ Ask Questions</h2>
                <div class="chat-messages" id="chatMessages">
                    <div class="message bot">👋 Upload a book to start</div>
                </div>
                <div class="input-area">
                    <input type="text" id="questionInput" placeholder="Ask a question..." onkeypress="event.key===\'Enter\' && askQuestion()">
                    <button class="btn" onclick="askQuestion()">Send</button>
                </div>
            </section>
        </div>
        <section class="card" id="sourcesSection" style="display:none; margin-top: 20px;">
            <h2>📌 Source References</h2>
            <div id="sourcesList"></div>
        </section>
    </main>

    <div class="pdf-modal" id="pdfModal">
        <div style="padding: 15px;">
            <div class="pdf-modal-header">
                <h3 id="pdfModalTitle">Page Preview</h3>
                <button class="pdf-modal-close" onclick="closePdfModal()">&times;</button>
            </div>
            <div class="pdf-modal-body">
                <div id="pdfContainer"></div>
            </div>
        </div>
    </div>
    <div class="pdf-overlay" id="pdfOverlay" onclick="closePdfModal()"></div>

    <script>
        const uploadArea = document.getElementById("uploadArea");
        const fileInput = document.getElementById("fileInput");
        let selectedFile = null;
        let currentPdfFile = null;
        
        uploadArea.addEventListener("click", () => fileInput.click());
        
        function handleDrop(e) {
            e.preventDefault();
            uploadArea.classList.remove("dragover");
            if (e.dataTransfer.files.length > 0) {
                selectedFile = e.dataTransfer.files[0];
                document.getElementById("fileName").textContent = selectedFile.name;
                document.getElementById("fileInfo").style.display = "block";
                document.getElementById("processBtn").style.display = "block";
            }
        }
        
        function handleFileSelect(e) {
            selectedFile = e.target.files[0];
            document.getElementById("fileName").textContent = selectedFile.name;
            document.getElementById("fileInfo").style.display = "block";
            document.getElementById("processBtn").style.display = "block";
        }
        
        async function processFile() {
            if (!selectedFile) return;
            document.getElementById("processBtn").disabled = true;
            showStatus("⏳ Processing PDF...", "processing");
            
            const formData = new FormData();
            formData.append("file", selectedFile);
            
            try {
                const response = await fetch("/api/upload", { method: "POST", body: formData });
                const data = await response.json();
                if (data.success) {
                    showStatus("✓ Book processed!", "success");
                    addMessage("bot", "✓ Ready to answer questions!");
                    currentPdfFile = selectedFile.name;
                } else {
                    showStatus("✗ Error: " + data.error, "error");
                }
            } catch (e) {
                showStatus("✗ Upload failed", "error");
            } finally {
                document.getElementById("processBtn").disabled = false;
            }
        }
        
        async function askQuestion() {
            const q = document.getElementById("questionInput").value.trim();
            if (!q) return;
            
            addMessage("user", q);
            document.getElementById("questionInput").value = "";
            
            try {
                const response = await fetch("/api/ask", {
                    method: "POST",
                    headers: { "Content-Type": "application/json" },
                    body: JSON.stringify({ question: q })
                });
                const data = await response.json();
                if (data.success) {
                    addMessage("bot", data.answer);
                    if (data.sources && data.sources.length > 0) {
                        displaySources(data.sources);
                    }
                } else {
                    addMessage("bot", "❌ Error: " + data.error);
                }
            } catch (e) {
                addMessage("bot", "❌ Request failed");
            }
        }
        
        function addMessage(type, content) {
            const msgs = document.getElementById("chatMessages");
            const msg = document.createElement("div");
            msg.className = "message " + type;
            msg.textContent = content;
            msgs.appendChild(msg);
            msgs.scrollTop = msgs.scrollHeight;
        }
        
        function showStatus(text, type) {
            const box = document.getElementById("statusBox");
            box.className = "status " + type;
            box.textContent = text;
        }
        
        function displaySources(sources) {
            const section = document.getElementById("sourcesSection");
            const list = document.getElementById("sourcesList");
            list.innerHTML = "";
            
            sources.forEach(source => {
                const card = document.createElement("div");
                card.className = "source-card";
                
                const pageTag = document.createElement("div");
                pageTag.className = "source-page";
                pageTag.textContent = "Page " + source.page;
                pageTag.onclick = (e) => {
                    e.stopPropagation();
                    if (currentPdfFile) {
                        openPdfPage(currentPdfFile, parseInt(source.page));
                    } else {
                        alert("PDF file not found");
                    }
                };
                
                const content = document.createElement("p");
                content.textContent = source.content;
                
                card.appendChild(pageTag);
                card.appendChild(content);
                list.appendChild(card);
            });
            
            section.style.display = "block";
        }
        
        function openPdfPage(filename, pageNum) {
            document.getElementById("pdfModal").classList.add("show");
            document.getElementById("pdfOverlay").classList.add("show");
            document.getElementById("pdfModalTitle").textContent = filename + " - Page " + pageNum;
            
            const pdfUrl = "/api/get-pdf/" + encodeURIComponent(filename);
            loadAndDisplayPdfPage(pdfUrl, pageNum);
        }
        
        function closePdfModal() {
            document.getElementById("pdfModal").classList.remove("show");
            document.getElementById("pdfOverlay").classList.remove("show");
            document.getElementById("pdfContainer").innerHTML = "";
        }
        
        async function loadAndDisplayPdfPage(pdfUrl, pageNum) {
            try {
                const pdf = await pdfjsLib.getDocument(pdfUrl).promise;
                
                if (pageNum > pdf.numPages || pageNum < 1) {
                    document.getElementById("pdfContainer").innerHTML = "<p style=\'color: #ef4444;\'>Page " + pageNum + " not found. PDF has " + pdf.numPages + " pages.</p>";
                    return;
                }
                
                const page = await pdf.getPage(pageNum);
                const viewport = page.getViewport({ scale: 1.5 });
                
                const canvas = document.createElement("canvas");
                canvas.width = viewport.width;
                canvas.height = viewport.height;
                
                const context = canvas.getContext("2d");
                await page.render({ canvasContext: context, viewport: viewport }).promise;
                
                document.getElementById("pdfContainer").innerHTML = "";
                document.getElementById("pdfContainer").appendChild(canvas);
            } catch (error) {
                document.getElementById("pdfContainer").innerHTML = "<p style=\'color: #ef4444;\'>Error: " + error.message + "</p>";
                console.error("PDF error:", error);
            }
        }
    </script>
</body>
</html>
    """
    return html

@app.route("/api/upload", methods=["POST"])
def upload_book():
    global qa_chain, vector_store_loaded
    try:
        if "file" not in request.files:
            return jsonify({"success": False, "error": "No file provided"}), 400
        
        file = request.files["file"]
        if not allowed_file(file.filename):
            return jsonify({"success": False, "error": "Only PDF files allowed"}), 400
        
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config["UPLOAD_FOLDER"], filename)
        file.save(filepath)
        
        print(f"Processing {filename}...")
        chunks = load_and_chunk_book(filepath)
        vector_store = create_vector_store(chunks)
        qa_chain = create_rag_chain(vector_store)
        vector_store_loaded = True
        
        return jsonify({"success": True, "chunks": len(chunks)})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

@app.route("/api/ask", methods=["POST"])
def ask_question():
    global qa_chain
    try:
        if qa_chain is None:
            return jsonify({"success": False, "error": "No book loaded"}), 400
        
        data = request.json
        question = data.get("question", "")
        result = qa_chain.invoke({"query": question})
        sources = []
        if "source_documents" in result:
            for doc in result["source_documents"]:
                sources.append({
                    "page": doc.metadata.get("page", "N/A"),
                    "content": doc.page_content[:200]
                })
        
        return jsonify({"success": True, "answer": result["result"], "sources": sources})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

@app.route("/api/get-pdf/<filename>")
def get_pdf(filename):
    """Serve uploaded PDF files"""
    try:
        file_path = os.path.join(app.config["UPLOAD_FOLDER"], filename)
        if os.path.exists(file_path) and filename.endswith(".pdf"):
            return send_file(file_path, mimetype="application/pdf")
        return {"error": "File not found"}, 404
    except Exception as e:
        return {"error": str(e)}, 500

if __name__ == "__main__":
    from pyngrok import ngrok
    public_url = ngrok.connect(5000)
    print(f"Public URL: {public_url}")
    app.run()
'''

with open('web_app_colab.py', 'w') as f:
    f.write(web_app_code)

print("✓ Flask web app created with PDF serving!")

## Step 6: Upload Your PDF (or use example)

In [ ]:
# Option 1: Upload a file from your computer
from google.colab import files
import os

print("Upload your PDF file:")
uploaded_files = files.upload()

if uploaded_files:
    for filename in uploaded_files.keys():
        print(f"✓ Uploaded: {filename}")
        # Move to uploads folder
        os.rename(filename, f'uploads/{filename}')
else:
    print("⚠️ No file uploaded. Using a sample PDF for demonstration...")
    # You can create a sample PDF or skip this

## Step 7: Start the Application with ngrok Tunnel

In [ ]:
import subprocess
import time
from pyngrok import ngrok

print("🚀 Starting Flask application with ngrok tunnel...\n")

# Create public URL
public_url = ngrok.connect(5000)
print(f"🌐 PUBLIC URL: {public_url}")
print(f"📱 Open this link in your browser to access the application")
print(f"\n{'='*60}\n")

# Start Flask app
subprocess.Popen(['python', 'web_app_colab.py'])
print("\n⏳ Waiting for server to start...")
time.sleep(3)
print("\n✓ Server is running!")
print("\n💡 Keep this cell running to maintain the ngrok tunnel.")
print("📝 To upload a PDF: Open the link above and use the upload area.")
print("❓ To ask questions: Use the Q&A section in the web interface.")

## Step 8: Quick Test (Optional)

In [ ]:
# Quick test of the RAG system
import os
from book_qa_colab import load_and_chunk_book, create_vector_store, create_rag_chain

# Find uploaded PDF
pdf_files = [f for f in os.listdir('uploads') if f.endswith('.pdf')]

if pdf_files:
    pdf_path = f'uploads/{pdf_files[0]}'
    print(f"Testing with: {pdf_files[0]}\n")
    
    # Process
    print("Loading and processing PDF...")
    chunks = load_and_chunk_book(pdf_path)
    print(f"✓ Created {len(chunks)} chunks\n")
    
    print("Creating vector store (this may take a moment)...")
    vector_store = create_vector_store(chunks)
    print("✓ Vector store ready\n")
    
    # Ask a test question
    print("Creating RAG chain...")
    qa_chain = create_rag_chain(vector_store)
    
    print("\n🤔 Asking a test question...\n")
    result = qa_chain.invoke({"query": "What is the main topic of this book?"})
    print(f"Answer: {result['result'][:500]}...")
    print(f"\n✓ RAG system is working!")
else:
    print("⚠️ No PDF files found in uploads folder.")
    print("Please upload a PDF using the web interface first.")

## Notes

- **GPU Usage**: The system uses the T4 GPU for LLM inference. Monitor GPU usage in the ngrok console.
- **Memory**: The Mistral-7B model uses ~7GB of GPU memory. Adjust if you run out of space.
- **Speed**: Expect 10-30 second response times depending on the question length.
- **Persistence**: The vector store is saved in `chroma_db/` and persists across sessions.
- **ngrok URL**: The public URL expires after 2 hours of inactivity. Restart the tunnel if needed.

For issues, check the server logs above this cell.